In [7]:
import pandas as pd

#load data from TSV

new_df= pd.read_csv('tsv/conga_multidft2_dropped.tsv', delimiter='\t')

display(new_df)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,0.000000,0.000000,0.00,0.00,0.000000,0.000000
1,71.428571,71.428571,0.03,0.10,0.481900,0.481900
2,128.571429,128.571429,0.03,0.10,0.525280,0.525280
3,157.142857,157.142857,0.03,0.10,0.556550,0.556550
4,200.000000,200.000000,0.03,0.10,0.567891,0.567891
...,...,...,...,...,...,...
65,395.000000,395.000000,0.20,0.40,0.051115,0.051115
66,515.000000,515.000000,0.20,0.40,0.021439,0.021439
67,550.000000,550.000000,0.20,0.40,0.018971,0.018971
68,600.000000,600.000000,0.20,0.40,0.043340,0.043340


In [8]:
# DataFrame utilities (works with both "single-frame" peak tables and time-window/event tables)
#
# Supported schemas:
# 1) Single-frame (peak list): columns = ["Frequency (Hz)", "Amplitude"]
# 2) Timeline / event table: columns = ["freq_start", "freq_stop", "time_start", "time_stop", "amp_min", "amp_max"]

import numpy as np
import pandas as pd


def _df_schema(df: pd.DataFrame) -> str:
    cols = set(df.columns)
    if {"Frequency (Hz)", "Amplitude"}.issubset(cols):
        return "frame"
    if {"freq_start", "freq_stop", "time_start", "time_stop", "amp_min", "amp_max"}.issubset(cols):
        return "events"
    raise ValueError(
        "Unsupported df schema. Expected either columns: "
        "['Frequency (Hz)', 'Amplitude'] or "
        "['freq_start','freq_stop','time_start','time_stop','amp_min','amp_max']."
    )


def normalize_amp(df: pd.DataFrame, *, method: str = "max", target: float = 1.0) -> pd.DataFrame:
    """Normalize amplitudes.

    - method='max': scale so max(|amp|) == target
    - method='sum': scale so sum(|amp|) == target
    - method='rms': scale so RMS(amp) == target
    """
    schema = _df_schema(df)
    out = df.copy()

    if schema == "frame":
        a = out["Amplitude"].astype(float).to_numpy()
        x = np.abs(a)
    else:
        a_min = out["amp_min"].astype(float).to_numpy()
        a_max = out["amp_max"].astype(float).to_numpy()
        x = np.abs(np.concatenate([a_min, a_max]))

    if method == "max":
        denom = float(np.max(x)) if x.size else 0.0
    elif method == "sum":
        denom = float(np.sum(x)) if x.size else 0.0
    elif method == "rms":
        denom = float(np.sqrt(np.mean(x * x))) if x.size else 0.0
    else:
        raise ValueError("method must be one of: 'max', 'sum', 'rms'")

    if not np.isfinite(denom) or denom <= 0:
        return out

    scale = float(target) / denom

    if schema == "frame":
        out["Amplitude"] = out["Amplitude"].astype(float) * scale
    else:
        out["amp_min"] = out["amp_min"].astype(float) * scale
        out["amp_max"] = out["amp_max"].astype(float) * scale

    return out


def pitch_scale(df: pd.DataFrame, factor: float = 1.0) -> pd.DataFrame:
    """Simple pitch scaling.

    - factor=1.0: unchanged
    - factor>1.0: higher pitch (frequencies multiplied)
    - factor<1.0: lower pitch
    """
    factor = float(factor)
    if not np.isfinite(factor) or factor <= 0:
        raise ValueError("factor must be finite and > 0")

    schema = _df_schema(df)
    out = df.copy()

    if schema == "frame":
        out["Frequency (Hz)"] = out["Frequency (Hz)"].astype(float) * factor
    else:
        out["freq_start"] = out["freq_start"].astype(float) * factor
        out["freq_stop"] = out["freq_stop"].astype(float) * factor

    return out


def time_stretch(df: pd.DataFrame, speed: float = 1.0) -> pd.DataFrame:
    """Time stretch by a *speed* factor.

    - speed=1.0: unchanged
    - speed>1.0: faster (times compressed)
    - speed<1.0: slower (times expanded)

    Note: Only applies to the event-table schema; for single-frame tables it returns df unchanged.
    """
    speed = float(speed)
    if not np.isfinite(speed) or speed <= 0:
        raise ValueError("speed must be finite and > 0")

    schema = _df_schema(df)
    out = df.copy()

    if schema == "events":
        scale = 1.0 / speed
        out["time_start"] = out["time_start"].astype(float) * scale
        out["time_stop"] = out["time_stop"].astype(float) * scale

    return out


# Example (uncomment):
df_norm = normalize_amp(new_df, method="max", target=1.0)
# df_pitch = pitch_scale(df_norm, factor=1.2)
df_stretch = time_stretch(df_norm, speed=0.25)
# display(df_fast)

display(df_stretch)

,freq_start,freq_stop,time_start,time_stop,amp_min,amp_max
0,0.000000,0.000000,0.00,0.0,0.000000,0.000000
1,71.428571,71.428571,0.12,0.4,0.481900,0.481900
2,128.571429,128.571429,0.12,0.4,0.525280,0.525280
3,157.142857,157.142857,0.12,0.4,0.556550,0.556550
4,200.000000,200.000000,0.12,0.4,0.567891,0.567891
...,...,...,...,...,...,...
65,395.000000,395.000000,0.80,1.6,0.051115,0.051115
66,515.000000,515.000000,0.80,1.6,0.021439,0.021439
67,550.000000,550.000000,0.80,1.6,0.018971,0.018971
68,600.000000,600.000000,0.80,1.6,0.043340,0.043340


In [9]:
from audiospylt.multiplotter import plot_combined, plot_combined_3d

# plot_combined(dfs=[df_stretch], df_labels=["stretch"])
plot_combined_3d(dfs=[df_stretch], df_labels=["stretch"],
    axis_order=("time", "amp", "freq"),
    flip={"time": True},
)  # if df has amp_min/amp_max too

In [10]:
from audiospylt.plot_wave import estimate_sampling_frequency_and_time_vector

sampling_frequency_updated, delta_t_updated, duration = estimate_sampling_frequency_and_time_vector(df_norm)

print(f"Estimated Optimal Sampling Frequency: {sampling_frequency_updated:.3f} Hz")
print(f"Sampling Interval (Delta t): {delta_t_updated:.6f} seconds")
print(f"Duration (highest value from time_stop column): {duration:.3f} seconds")


Estimated Optimal Sampling Frequency: 5600.000 Hz
Sampling Interval (Delta t): 0.000179 seconds
Duration (highest value from time_stop column): 0.750 seconds


In [15]:
from audiospylt.plot_wave import plot_waves

# define the time vector (from 0 to n seconds, with a step of k seconds)
k=0.00005
sample_rate = 1.0 / k

# Optional FFT/spectrogram settings (forwarded to py_scripts.audio_utils.plot_spectrogram)
spectrogram_kwargs = {
    # y-axis scaling
    "y_axis_mode": "log",         # 'linear' | 'log' | 'mel' | 'mixed'
    "y_axis_mix": 0.5,              # 0..1 (only for 'mixed')
    "mixed_log_floor_hz": 1.0,      # >0 (only for 'mixed')

    # FFT/time-frequency settings
    "n_fft": 512,                  # FFT size; larger -> finer freq, coarser time
    "window_type": "hann",         # window shape
    "overlap": 0.75,                 # keep <0.95
    "oversample_factor": 1.0,       # >=1.0; 2.0 pads FFT

    # mel settings (only for y_axis_mode='mel')
    "mel_bins": 128,
    "mel_fmax": sample_rate / 2,    # cap at Nyquist

    # rendering
    "scaling": "density",          # 'density' | 'spectrum'
    "mode": "magnitude",           # 'magnitude' | 'psd'
    "cmap": "Magma",
    "boundary": "zeros",
    "padded": True,
    "time_range": "signal",
    "show": True,                 # display figure
}

# Default behavior: show synthesized waveform only (k-resolution intact).
# Flip show_fft=True when you want the spectrogram.
y_combined = plot_waves(
    df_stretch,
    k,
    edge_fade_s=0,      
    phase_mode="global",    # helps continuity for freq_start==freq_stop bins
    show_waveform=True,
    show_fft=True,
    spectrogram_kwargs=spectrogram_kwargs,
)

In [16]:
from audiospylt.generate_wave_file import render_audio

player = True
save_audio = True

fs_initial = 1 / k  # The initial sampling rate is the inverse of the time step (k)

render_audio(
    y_combined,
    fs_initial,
    fs_target_name="44.1kHz",
    bit_rate=24,
    filename_template="testing_{fs_target_name}_{bit_rate}bit_{timestamp}",
    timestamp_format="%Y-%m-%d_%H-%M-%S",
    save_audio=save_audio,
    player=player,
    sanitize=True,
    verbose=True,
)


[2026-01-04 03:30:57] 24-bit wave file with 44.1kHz sampling rate saved successfully to: c:\Users\egorp\Nextcloud\code\public_repos\audiospylt\notebooks\rendered_audio\testing_44.1kHz_24bit_2026-01-04_03-30-57.wav


'c:\\Users\\egorp\\Nextcloud\\code\\public_repos\\audiospylt\\notebooks\\rendered_audio\\testing_44.1kHz_24bit_2026-01-04_03-30-57.wav'